# PhishGuard v2 — Retraining Data-Centric AI (Enriched Live Feeds + Out-of-Time Holdout Evaluation)
### Google Colab Notebook — Kompatibel 100% dengan Pipeline 27 Fitur & Backend Inferensi PhishGuard v2

Notebook ini dirancang khusus untuk dijalankan di **Google Colab** (`Runtime -> Run all`) guna melatih ulang (**retrain dari awal**) model **PhishGuard v2** dengan dataset yang jauh lebih kaya dan evaluasi yang jujur (*anti-leakage* & *out-of-time*):

1. **Mempertahankan Kontrak Arsitektur v2 (27 Fitur + Stacked TF-IDF):**
   - Menjaga kompatibilitas penuh dengan `backend/app/services/feature_extraction.py`, `prediction.py`, dan `tests/` (`26` fitur tabular/fuzzy + `tfidf_char_prob`).
2. **Pengayaan Dataset Latih (*Data-Centric Augmentation + Multi-Feed*):**
   - **Dataset Utama:** **PhiUSIIL** (235k+ URL), di-*split* lebih dulu (`70% Train / 15% Val / 15% Test`) sehingga `test_df` 100% bersih.
   - **Phishing Nyata Modern:** Mengambil feed aktif **OpenPhish** dan **PhishTank (`online-valid.csv`)**, diprioritaskan pada **phishing ber-HTTPS**, **sub-domain cloud/hosting gratis** (`*.pages.dev`, `*.vercel.app`, `*.workers.dev`, `*.firebaseapp.com`, `*.blogspot.com`, `*.bolt.host`, dll.), **host IP mentah**, serta TLD mencurigakan.
   - **Phishing Lokal Indonesia & Adversaria:** Variasi umpan lokal (`dana-kaget`, `bansos`, `brimo`, `klikbca`, `mybca`, `prakerja`, `blt`, `ojk`), typosquatting, homograph, dan IP-host.
   - **URL Aman Dunia Nyata (*Legitimate De-biasing*):** Memanfaatkan domain nyata dari daftar **Tranco Top Sites** (disaring agar bebas platform *shared hosting*/shortener) + portal kampus/pemerintah/berita/dokumentasi dengan proporsi **HTTP (~35–40%) & HTTPS (~60–65%)**, path dalam, multi-titik `.html#anchor`, serta parameter `?utm_source=...` — dirancang **disjoint** (domain terpisah) dari set uji agar model tidak menghafal.
3. **Evaluasi Ketat & *Out-of-Time Holdout*:**
   - **Evaluasi 1:** Clean Held-Out Test Set **PhiUSIIL** (komparasi adil: Baseline LR vs RF v1 vs PhishGuard v2 Enriched).
   - **Evaluasi 2:** **Out-of-Time Real-World Holdout** — URL PhishTank terbaru secara kronologis + 50% host OpenPhish terpisah + URL aman Tranco/dokumentasi/berita/bank yang **belum pernah dilihat** selama training.
   - **Evaluasi 3:** **Stress Test Murni ML** (`10` kasus klasik + `20` kasus diperluas tanpa whitelist/aturan `if-else`) serta komparasi **Permutation Feature Importance** pada PhiUSIIL vs Out-of-Time Holdout.


In [ ]:
# [1] Setup Dependensi & Konfigurasi Google Colab
import os
import re
import io
import csv
import glob
import math
import json
import zipfile
import warnings
import urllib.request
from collections import Counter
from difflib import SequenceMatcher
from urllib.parse import urlsplit
from concurrent.futures import ThreadPoolExecutor

warnings.filterwarnings("ignore")

import joblib
import sklearn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

try:
    import lightgbm as lgb
    from lightgbm import LGBMClassifier
    USE_LIGHTGBM = True
    V2_ALGO_NAME = "LightGBM (LGBMClassifier)"
    LGB_VER = lgb.__version__
except ImportError:
    USE_LIGHTGBM = False
    V2_ALGO_NAME = "HistGradientBoostingClassifier"
    LGB_VER = "N/A"

print(f"✓ Lingkungan Siap | Scikit-Learn: {sklearn.__version__} | LightGBM: {LGB_VER} | Engine v2: {V2_ALGO_NAME}")

In [ ]:
# [2] Akuisisi Dataset Utama (PhiUSIIL) + Feed Phishing Aktif (OpenPhish & PhishTank) + Domain Tranco

UA_HEADER = {"User-Agent": "PhishGuard-Research/2.1"}

def _http_get_bytes(url: str, timeout: int = 30) -> bytes:
    req = urllib.request.Request(url, headers=UA_HEADER)
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return resp.read()

# 1. Muat PhiUSIIL
dataset_df = None
dataset_source = "unknown"

local_csvs = (
    glob.glob("/content/**/*.csv", recursive=True)
    + glob.glob("dataset/**/*.csv", recursive=True)
)
phiusiil_local = [p for p in local_csvs if "phiusiil" in os.path.basename(p).lower()]

if phiusiil_local:
    print(f"✓ Menemukan dataset PhiUSIIL lokal/Drive: {phiusiil_local[0]}")
    dataset_df = pd.read_csv(phiusiil_local[0], usecols=lambda c: c in {"URL", "label"})
    dataset_source = "phiusiil"
else:
    print("ℹ Mengunduh PhiUSIIL URL Dataset resmi dari UCI ML Repository...")
    UCI_ZIP_URL = "https://archive.ics.uci.edu/static/public/967/phiusiil+phishing+url+dataset.zip"
    ZIP_PATH = "phiusiil_dataset.zip"
    EXTRACT_DIR = "dataset_extracted"
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    try:
        if not glob.glob(f"{EXTRACT_DIR}/**/*.csv", recursive=True):
            with open(ZIP_PATH, "wb") as out_f:
                out_f.write(_http_get_bytes(UCI_ZIP_URL, timeout=120))
            with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
                zip_ref.extractall(EXTRACT_DIR)
        extracted_csv = glob.glob(f"{EXTRACT_DIR}/**/*.csv", recursive=True)
        if extracted_csv:
            dataset_df = pd.read_csv(extracted_csv[0], usecols=lambda c: c in {"URL", "label"})
            dataset_source = "phiusiil"
            print(f"✓ Sukses memuat PhiUSIIL: {len(dataset_df):,} baris data!")
    except Exception as e:
        print(f"⚠️ Gagal mengunduh UCI zip ({e}), menggunakan dataset alternatif...")
        ALT_URL = "https://raw.githubusercontent.com/datasets/phishing-urls/main/data/dataset.csv"
        dataset_df = pd.read_csv(ALT_URL)
        dataset_source = "alternative"

url_col = "URL" if "URL" in dataset_df.columns else dataset_df.columns[0]
label_col = "label" if "label" in dataset_df.columns else dataset_df.columns[1]
dataset_df = dataset_df[[url_col, label_col]].dropna().drop_duplicates(subset=[url_col]).copy()

if dataset_source == "phiusiil" or len(dataset_df) > 200000:
    dataset_df["target"] = (dataset_df[label_col].astype(int) == 0).astype(int)  # 1 = Phishing, 0 = Legitimate
else:
    dataset_df["target"] = (dataset_df[label_col].astype(int) == 1).astype(int)
dataset_df = dataset_df.rename(columns={url_col: "URL"})[["URL", "target"]].reset_index(drop=True)
print(f"✓ Total URL unik PhiUSIIL siap olah: {len(dataset_df):,} ({dataset_df['target'].mean()*100:.1f}% Phishing)")

# 2. Tarik OpenPhish (Live Feed)
openphish_urls = []
try:
    raw_op = _http_get_bytes("https://openphish.com/feed.txt", timeout=15).decode("utf-8", errors="ignore")
    openphish_urls = list(dict.fromkeys(
        u.strip() for u in raw_op.splitlines() if u.strip().startswith(("http://", "https://"))
    ))
    print(f"✓ OpenPhish live feed: {len(openphish_urls):,} URL phishing aktif")
except Exception as e:
    print(f"ℹ OpenPhish tidak dapat ditarik ({e}), lanjut tanpa OpenPhish.")

# 3. Tarik PhishTank Verified Online CSV (Live + Historical Cronological Feed)
phishtank_rows = []
try:
    raw_pt = _http_get_bytes("https://data.phishtank.com/data/online-valid.csv", timeout=35).decode("utf-8", errors="ignore")
    reader = csv.DictReader(io.StringIO(raw_pt))
    seen_pt = set()
    for row in reader:
        u = str(row.get("url", "")).strip()
        verified = str(row.get("verified", "")).strip().lower()
        sub_time = str(row.get("submission_time", "")).strip()
        if verified == "yes" and u.startswith(("http://", "https://")) and len(u) <= 2048 and u not in seen_pt:
            seen_pt.add(u)
            phishtank_rows.append({"url": u, "submission_time": sub_time})
    # Urutkan terbaru -> terlama secara kronologis untuk pemisahan Out-of-Time yang valid
    phishtank_rows.sort(key=lambda r: r["submission_time"], reverse=True)
    https_pt = sum(1 for r in phishtank_rows if r["url"].startswith("https://"))
    print(f"✓ PhishTank verified online: {len(phishtank_rows):,} URL ({https_pt:,} HTTPS / {https_pt/max(1,len(phishtank_rows))*100:.1f}%)")
except Exception as e:
    print(f"ℹ PhishTank tidak dapat ditarik ({e}), lanjut dengan feed lain.")

# 4. Tarik Tranco Top 20,000 Real Benign Domains
tranco_domains = []
try:
    latest_meta = json.loads(_http_get_bytes("https://tranco-list.eu/api/lists/date/latest", timeout=15).decode("utf-8"))
    list_id = latest_meta.get("list_id")
    if list_id:
        raw_tr = _http_get_bytes(f"https://tranco-list.eu/download/{list_id}/20000", timeout=30).decode("utf-8", errors="ignore")
        for line in raw_tr.splitlines():
            parts = line.strip().split(",", 1)
            if len(parts) == 2 and "." in parts[1]:
                tranco_domains.append(parts[1].strip().lower())
    print(f"✓ Tranco Top Domains ({list_id}): {len(tranco_domains):,} domain sah terverifikasi")
except Exception as e:
    print(f"ℹ Tranco tidak dapat ditarik ({e}), menggunakan korpus domain sah cadangan.")

In [ ]:
# [3] Feature Engineering v2 (Sinkron 100% dengan backend/app/services/feature_extraction.py)

SUSPICIOUS_KEYWORDS = [
    "login", "verify", "account", "secure", "update", "signin",
    "banking", "password", "wallet", "confirm", "recover", "suspended",
    "webscr", "ebayisapi", "paypal", "credential"
]
INDO_KEYWORDS = [
    "undian", "bansos", "prakerja", "dana", "dana-kaget", "pulsa",
    "kejutan", "hadiah", "bca", "bri", "bni", "mandiri", "gopay",
    "shopee", "linkaja", "kuota", "subsidi", "blt", "ojk", "kominfo",
    "klikbca", "brimo"
]
URL_SHORTENERS = {
    "bit.ly", "tinyurl.com", "t.co", "goo.gl", "is.gd", "ow.ly",
    "buff.ly", "rebrand.ly", "cutt.ly", "shorturl.at", "tiny.cc",
    "lnkd.in", "s.id", "rb.gy", "v.gd", "clck.ru", "rotf.lol",
    "trib.al", "tny.im"
}
SUSPICIOUS_TLDS = {
    "xyz", "top", "tk", "ml", "ga", "cf", "gq", "work", "click",
    "buzz", "fit", "rest", "cam", "kim", "bar", "icu", "club",
    "live", "surf", "monster", "cfd"
}
CONFUSABLES = set("\u0430\u0441\u0435\u043e\u0440\u0455\u0445\u0456\u0458\u0443")

OFFICIAL_BRANDS = {
    "google": ["google.com", "google.co.id", "gmail.com"],
    "bca": ["bca.co.id", "klikbca.com"],
    "bri": ["bri.co.id", "ib.bri.co.id"],
    "mandiri": ["bankmandiri.co.id", "livinbymandiri.id"],
    "bni": ["bni.co.id"],
    "dana": ["dana.id"],
    "gopay": ["gopay.co.id", "gojek.com"],
    "shopee": ["shopee.co.id", "shopeepay.co.id"],
    "tokopedia": ["tokopedia.com"],
    "paypal": ["paypal.com"],
    "microsoft": ["microsoft.com", "live.com", "office.com"],
    "apple": ["apple.com", "icloud.com"],
    "detik": ["detik.com"],
    "kompas": ["kompas.com"],
}
OFFICIAL_DOMAINS_FLAT = {d for ds in OFFICIAL_BRANDS.values() for d in ds}
LEET_TABLE = str.maketrans("01345", "oieas")


def calculate_shannon_entropy(text: str) -> float:
    if not text:
        return 0.0
    n = len(text)
    counts = Counter(text)
    return round(-sum((c / n) * math.log2(c / n) for c in counts.values()), 3)


def strip_scheme(url: str) -> str:
    """Menghilangkan prefix http:// atau https:// agar TF-IDF fokus ke struktur domain dan path."""
    return re.sub(r"^https?://", "", str(url).strip(), flags=re.IGNORECASE)


def compute_brand_fuzzy(host: str, full_lower: str) -> tuple[float, int, int]:
    """Menghitung kemiripan typosquatting (termasuk leetspeak) terhadap merek resmi."""
    if not host:
        return 0.0, 0, 0
    h = host[4:] if host.startswith("www.") else host
    is_official = int(any(h == d or h.endswith("." + d) for d in OFFICIAL_DOMAINS_FLAT))
    if is_official:
        return 0.0, 1, 0

    host_tokens = [t for t in re.split(r"[.\-_]", h) if len(t) >= 3]
    best_sim = 0.0
    brand_abuse = 0
    for brand_key in OFFICIAL_BRANDS:
        if brand_key in full_lower:
            brand_abuse = 1
        blen = len(brand_key)
        for tok in host_tokens:
            if abs(len(tok) - blen) > 3:
                continue
            norm_tok = tok.translate(LEET_TABLE)
            sim = max(
                SequenceMatcher(None, tok, brand_key).ratio(),
                SequenceMatcher(None, norm_tok, brand_key).ratio(),
            )
            if sim > best_sim:
                best_sim = sim
    return round(best_sim, 3), 0, brand_abuse


def extract_v2_features(url_str: str) -> dict:
    raw = str(url_str).strip()
    lowered = raw.lower()
    try:
        parsed = urlsplit(raw if "://" in raw else f"//{raw}")
        host = (parsed.hostname or "").rstrip(".").lower()
        port = parsed.port
        path = parsed.path or ""
        query = parsed.query or ""
    except (ValueError, UnicodeError):
        host, port, path, query = "", None, "", ""

    parts = host.split(".")
    is_ip = 1 if len(parts) == 4 and all(p.isdigit() and 0 <= int(p) <= 255 for p in parts) else 0
    subdomain_count = max(0, len([p for p in parts if p]) - 2)
    susp_hits = sum(lowered.count(kw) for kw in SUSPICIOUS_KEYWORDS)
    indo_hits = sum(lowered.count(kw) for kw in INDO_KEYWORDS)
    has_homograph = 1 if ("xn--" in host or any(c in CONFUSABLES for c in host)) else 0
    tld = parts[-1] if parts else ""
    is_suspicious_tld = 1 if tld in SUSPICIOUS_TLDS else 0
    is_official_tld = 1 if (host.endswith(".ac.id") or host.endswith(".go.id") or tld in {"edu", "gov"}) else 0
    is_shortener = 1 if any(host == s or host.endswith("." + s) for s in URL_SHORTENERS) else 0
    has_port = 1 if port is not None else 0
    host_entropy = calculate_shannon_entropy(host)
    path_entropy = calculate_shannon_entropy(path)
    typosquat_sim, is_official_brand, has_brand_abuse = compute_brand_fuzzy(host, lowered)

    return {
        # 20 Fitur Leksikal v1
        "url_length": len(raw),
        "hostname_length": len(host),
        "path_length": len(path),
        "count_dot": raw.count("."),
        "count_hyphen": raw.count("-"),
        "count_at": raw.count("@"),
        "count_question": raw.count("?"),
        "count_equal": raw.count("="),
        "count_slash": raw.count("/"),
        "count_digits": sum(c.isdigit() for c in raw),
        "count_subdomains": subdomain_count,
        "has_https": 1 if lowered.startswith("https://") else 0,
        "has_ip": is_ip,
        "has_port": has_port,
        "shannon_entropy": host_entropy,
        "has_homograph": has_homograph,
        "is_suspicious_tld": is_suspicious_tld,
        "has_url_shortener": is_shortener,
        "suspicious_words": susp_hits,
        "indonesian_keywords": indo_hits,
        # 6 Fitur Tambahan v2
        "query_length": len(query),
        "path_entropy": path_entropy,
        "is_official_tld": is_official_tld,
        "typosquat_similarity": typosquat_sim,
        "is_official_brand": is_official_brand,
        "has_brand_abuse": has_brand_abuse,
    }


V1_FEATURE_COLS = [
    "url_length", "hostname_length", "path_length", "count_dot", "count_hyphen",
    "count_at", "count_question", "count_equal", "count_slash", "count_digits",
    "count_subdomains", "has_https", "has_ip", "has_port", "shannon_entropy",
    "has_homograph", "is_suspicious_tld", "has_url_shortener", "suspicious_words",
    "indonesian_keywords"
]
V2_TABULAR_COLS = V1_FEATURE_COLS + [
    "query_length", "path_entropy", "is_official_tld",
    "typosquat_similarity", "is_official_brand", "has_brand_abuse"
]
V2_ALL_COLS = V2_TABULAR_COLS + ["tfidf_char_prob"]
print(f"✓ Konfigurasi Fitur Siap: v1 ({len(V1_FEATURE_COLS)} fitur) -> v2 ({len(V2_ALL_COLS)} fitur, sinkron dengan backend)")

In [ ]:
# [4] Stratified Split + Pemisahan Strict Train vs Out-of-Time Holdout (Zero Data Leakage)

# 1. Split PhiUSIIL lebih dulu agar val_df & test_df 100% murni PhiUSIIL
train_df, temp_df = train_test_split(dataset_df, test_size=0.30, random_state=42, stratify=dataset_df["target"])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df["target"])

def _extract_host(u: str) -> str:
    try:
        h = (urlsplit(u if "://" in u else f"//{u}").hostname or "").lower().rstrip(".")
        return h[4:] if h.startswith("www.") else h
    except Exception:
        return ""

def _is_ipv4(host: str) -> bool:
    p = host.split(".")
    return len(p) == 4 and all(x.isdigit() and 0 <= int(x) <= 255 for x in p)

# Domain/URL yang HARUS disisihkan dari data latih tambahan (PhiUSIIL Val/Test + Stress Test Holdout)
RESERVED_STRESS_HOSTS = {
    "liputan6.com", "akademik-stikom.ac.id", "bri.co.id", "docs.python.org", "python.org",
    "dana-kaget-gratis-2026.xyz", "cek-penerima-blt-kemensos.top", "mybca-aktivasi-ulang.com",
    "g00gle-security-alert.com", "paypa1-resolution-center.net", "xn--pple-43d.com",
    "bni.co.id", "ui.ac.id", "its.ac.id", "kemdikbud.go.id", "tempo.co", "cnbcindonesia.com",
}

phi_eval_urls = set(val_df["URL"]).union(set(test_df["URL"]))
phi_eval_hosts = {_extract_host(u) for u in phi_eval_urls if _extract_host(u)} | RESERVED_STRESS_HOSTS

CLOUD_ABUSE_SUFFIXES = (
    "pages.dev", "workers.dev", "vercel.app", "netlify.app", "firebaseapp.com",
    "web.app", "blogspot.com", "sites.google.com", "bolt.host", "weebly.com",
    "github.io", "gitlab.io", "wixsite.com", "herokuapp.com", "replit.app",
    "myfreesites.net", "webflow.io", "framer.website", "glitch.me"
)

def _is_cloud_subdomain(host: str) -> bool:
    return any(host == s or host.endswith("." + s) for s in CLOUD_ABUSE_SUFFIXES)

def _is_shared_or_shortener(domain: str) -> bool:
    return _is_cloud_subdomain(domain) or any(domain == s or domain.endswith("." + s) for s in URL_SHORTENERS)

# 2. Pisahkan Feed Nyata Phishing -> Out-of-Time Holdout vs Train Feed (Disjoint Host!)
oot_phish_rows = []
oot_phish_hosts = set()
oot_phish_urls = set()

# 2a. Ambil max 1,500 URL terbaru dari PhishTank untuk Out-of-Time Holdout
for r in phishtank_rows:
    u = r["url"]
    h = _extract_host(u)
    if not h or u in phi_eval_urls or h in phi_eval_hosts or u in oot_phish_urls:
        continue
    if len(oot_phish_rows) < 1500:
        oot_phish_rows.append((u, 1))
        oot_phish_urls.add(u)
        oot_phish_hosts.add(h)
    else:
        break

# 2b. Split OpenPhish 50% OOT Holdout vs 50% Train (strict disjoint by host!)
op_clean = []
for u in openphish_urls:
    h = _extract_host(u)
    if h and u not in phi_eval_urls and h not in phi_eval_hosts and h not in oot_phish_hosts:
        op_clean.append((u, h))

op_unique_hosts = list(dict.fromkeys(h for _, h in op_clean))
oot_op_hosts = set(op_unique_hosts[::2])
for u, h in op_clean:
    if h in oot_op_hosts and len(oot_phish_rows) < 1650 and u not in oot_phish_urls:
        oot_phish_rows.append((u, 1))
        oot_phish_urls.add(u)
        oot_phish_hosts.add(h)
oot_phish_hosts |= oot_op_hosts

op_train_urls = [u for u, h in op_clean if h not in oot_phish_hosts]

# 2c. Ambil hingga 12,000 PhishTank terverifikasi (beragam HTTPS / Cloud / IP / TLD) untuk Train
train_pt_https, train_pt_cloud, train_pt_ip, train_pt_other = [], [], [], []
for r in phishtank_rows:
    u = r["url"]
    h = _extract_host(u)
    if not h or u in phi_eval_urls or h in phi_eval_hosts or u in oot_phish_urls or h in oot_phish_hosts:
        continue
    if _is_ipv4(h):
        train_pt_ip.append(u)
    elif _is_cloud_subdomain(h):
        train_pt_cloud.append(u)
    elif u.startswith("https://"):
        train_pt_https.append(u)
    else:
        train_pt_other.append(u)

feed_train_phish = list(dict.fromkeys(
    op_train_urls
    + train_pt_cloud[:3000]
    + train_pt_ip[:1000]
    + train_pt_https[:6500]
    + train_pt_other[:1500]
))
all_phish_hosts = oot_phish_hosts | {_extract_host(u) for u in feed_train_phish if _extract_host(u)}

# 3. Bangun Korpus Tranco + Template Aman Dunia Nyata (Strict Disjoint Train vs OOT Holdout)
rng = np.random.default_rng(42)

train_edu_gov = [
    "unpad.ac.id", "ugm.ac.id", "itb.ac.id", "undip.ac.id", "ipb.ac.id",
    "unair.ac.id", "usu.ac.id", "unhas.ac.id", "uny.ac.id", "binus.ac.id",
    "kemkes.go.id", "kominfo.go.id", "pajak.go.id", "kemenkeu.go.id", "bssn.go.id",
]
train_news_doc_bank = [
    "kompas.com", "detik.com", "kumparan.com", "antaranews.com", "cnnindonesia.com",
    "sindonews.com", "republika.co.id", "bca.co.id", "bankmandiri.co.id", "bsi.co.id",
    "tokopedia.com", "gojek.com", "traveloka.com", "pandas.pydata.org", "numpy.org",
    "scikit-learn.org", "fastapi.tiangolo.com", "kubernetes.io", "mozilla.org", "apache.org",
]
oot_benign_hosts_extra = ["its.ac.id", "ui.ac.id", "kemdikbud.go.id", "tempo.co", "cnbcindonesia.com", "rust-lang.org"]
reserved_curated_doms = set(train_edu_gov) | set(train_news_doc_bank) | set(oot_benign_hosts_extra)

clean_tranco = []
seen_tr = set()
for d in tranco_domains:
    d_base = d[4:] if d.startswith("www.") else d
    if (
        d_base
        and d_base not in seen_tr
        and d_base not in phi_eval_hosts
        and d_base not in all_phish_hosts
        and d_base not in reserved_curated_doms
        and not _is_shared_or_shortener(d_base)
    ):
        seen_tr.add(d_base)
        clean_tranco.append(d_base)

# Jika Tranco offline, gunakan fallback daftar domain sah yang terpisah dari OOT
if len(clean_tranco) < 200:
    fallback_doms = [
        "wikipedia.org", "github.com", "kernel.org", "debian.org", "golang.dev",
        "reactjs.org", "vuejs.org", "w3.org", "readthedocs.io", "ietf.org"
    ]
    clean_tranco = [d for d in fallback_doms if d not in phi_eval_hosts and d not in all_phish_hosts and d not in reserved_curated_doms]

# Pisah domain Tranco secara ketat: ~15% untuk OOT Benign Holdout, ~85% untuk Train
tranco_oot_doms = [d for i, d in enumerate(clean_tranco) if i % 7 == 0][:1500]
tranco_train_doms = [d for i, d in enumerate(clean_tranco) if i % 7 != 0]

paths_a = ["library", "reference", "guide", "tutorial", "article", "archive", "modul", "akademik", "kurikulum", "berita", "tekno", "riset", "sistem", "produk", "layanan", "dokumen"]
paths_b = ["overview", "getting-started", "v2", "index", "detail", "panduan", "pengumuman", "dokumentasi", "chapter-1", "release-notes", "individu", "simpanan", "api", "profil"]
doc_files = ["manual.html", "spec.html", "view.php", "guide.pdf", "overview.md", "report.html", "index.jsp", "module.api.html", "urllib.request.html", "std.collection.html"]

def make_realistic_benign_url(dom: str, idx: int, rng_obj) -> str:
    mode = idx % 5
    pa, pb = rng_obj.choice(paths_a), rng_obj.choice(paths_b)
    fn = rng_obj.choice(doc_files)
    use_http = (idx % 10) < 4  # ~40% HTTP aman, ~60% HTTPS aman untuk mengikis bias has_https!
    sch = "http" if use_http else "https"
    prefix = "" if dom.count(".") >= 2 or dom.startswith("www.") else "www."
    if mode == 0:
        return f"{sch}://{prefix}{dom}/{pa}/{pb}/{100 + (idx % 900)}"
    elif mode == 1:
        sub = "docs" if not dom.startswith(("docs.", "api.", "wiki.")) else ""
        host_d = f"{sub}.{dom}" if sub else dom
        return f"{sch}://{host_d}/{(idx % 5) + 1}.{idx % 12}/{pa}/{fn}#{pa}-{pb}"
    elif mode == 2:
        return f"{sch}://{prefix}{dom}/{pa}/read/2026/{(idx % 12) + 1:02d}/{10000 + idx}/{pb}?utm_source=share&utm_medium=social&ref=home"
    elif mode == 3:
        return f"{sch}://{prefix}{dom}/id/{pa}/{pb}?lang=id&page={(idx % 20) + 1}"
    else:
        return f"{sch}://{prefix}{dom}/"

# 3a. Bangun Out-of-Time Benign Holdout (~1,650 URL dari domain yang TIDAK PERNAH masuk train)
oot_benign_pool = tranco_oot_doms + [d for d in oot_benign_hosts_extra if d not in all_phish_hosts]
oot_benign_rows = []
for i in range(1650):
    dom = oot_benign_pool[i % len(oot_benign_pool)]
    oot_benign_rows.append((make_realistic_benign_url(dom, i, rng), 0))

# Pastikan OOT tidak memunculkan host Tranco milik train, dan buang baris OpenPhish yang kebetulan beririsan domain Tranco
oot_df = pd.DataFrame(oot_phish_rows + oot_benign_rows, columns=["URL", "target"]).drop_duplicates(subset=["URL"]).reset_index(drop=True)
oot_all_hosts = {_extract_host(u) for u in oot_df["URL"] if _extract_host(u)}

# 3b. Bangun Train Augmentation Legitimate (~18,000 URL) + Adversarial Phishing (~10,000 URL) + Live Feeds (~12,150 URL)
aug_rows = []
train_benign_pool = list(dict.fromkeys(tranco_train_doms[:12000] + train_edu_gov * 15 + train_news_doc_bank * 15))
train_benign_pool = [d for d in train_benign_pool if d not in oot_all_hosts]

n_legit_aug = 18000
for i in range(n_legit_aug):
    dom = train_benign_pool[i % len(train_benign_pool)]
    u_ben = make_realistic_benign_url(dom, i + 7, rng)
    if _extract_host(u_ben) not in oot_all_hosts:
        aug_rows.append((u_ben, 0))

# Phishing Adversarial Tambahan (Indo lures HTTPS + Typosquatting + Cloud Subdomains + IP Host + Homograph)
phish_lures = ["dana-kaget", "bansos", "prakerja", "brimo", "klikbca", "blt", "hadiah", "undian", "subsidi", "ojk-blokir", "mybca-aktivasi", "kuota-gratis"]
phish_brands = ["bcaa", "paypa1", "g00gle", "mandlri", "sh0pee", "app1e", "m1crosoft", "netf1ix", "te1egram", "whats4pp", "brii", "dan4"]
phish_actions = ["login", "verify", "update", "signin", "confirm", "daftar", "klaim", "ambil", "auth", "secure"]
bad_tlds = ["xyz", "top", "click", "cfd", "tk", "icu", "work", "buzz"]
cloud_hosts = ["pages.dev", "workers.dev", "vercel.app", "netlify.app", "firebaseapp.com", "web.app", "blogspot.com", "sites.google.com"]

n_adv_phish = 10000
for i in range(n_adv_phish):
    mode = i % 5
    act = rng.choice(phish_actions)
    if mode == 0:  # Phishing HTTPS lokal Indonesia
        u = f"https://{rng.choice(phish_lures)}-{rng.choice(['resmi', 'gratis', '2026', 'kilat', 'resmi-id'])}-{10 + (i % 900)}.{rng.choice(bad_tlds)}/{act}"
    elif mode == 1:  # Typosquatting brand HTTPS
        u = f"https://{rng.choice(phish_brands)}-{rng.choice(['security', 'support', 'alert', 'verify', 'center', 'online'])}-{10 + (i % 500)}.{rng.choice(['com', 'net', 'xyz', 'top'])}/{act}"
    elif mode == 2:  # Subdomain layanan cloud gratis (HTTPS)
        u = f"https://{rng.choice(phish_lures)}-{rng.choice(['portal', 'verif', 'resmi'])}-{10 + (i % 800)}.{rng.choice(cloud_hosts)}/{act}.html"
    elif mode == 3:  # IP-host phishing (HTTP & HTTPS dengan/tanpa port)
        oct1 = rng.choice([45, 91, 103, 149, 178, 185, 194, 203])
        ip_str = f"{oct1}.{rng.integers(10, 240)}.{rng.integers(1, 250)}.{rng.integers(2, 250)}"
        sch = "http" if i % 2 == 0 else "https"
        port_s = f":{rng.choice([8080, 8443, 2083])}" if i % 3 == 0 else ""
        u = f"{sch}://{ip_str}{port_s}/{rng.choice(['auth', 'banking', 'signin', 'webscr'])}/{act}.php?id={1000 + i}"
    else:  # IDN Homograph HTTPS
        u = f"https://xn--{rng.choice(['pple', 'gogle', 'bca', 'bri', 'pypal'])}-{10 + (i % 80)}a.com/{act}"
    if _extract_host(u) not in oot_all_hosts and _extract_host(u) not in phi_eval_hosts:
        aug_rows.append((u, 1))

# Gabungkan Live Feed Phishing (PhishTank Train + OpenPhish Train) dengan filter disjoint host
feed_train_phish = [u for u in feed_train_phish if _extract_host(u) not in oot_all_hosts and _extract_host(u) not in phi_eval_hosts]
for live_u in feed_train_phish:
    aug_rows.append((live_u, 1))

aug_df = pd.DataFrame(aug_rows, columns=["URL", "target"]).drop_duplicates(subset=["URL"]).reset_index(drop=True)
train_aug_df = pd.concat([train_df, aug_df], ignore_index=True)

# Bobot 2.5x pada korpus terkurasi + live feed agar bias PhiUSIIL teratasi tanpa mengorbankan akurasi global
aug_weight = 2.5
sample_weights_v2 = np.concatenate([np.ones(len(train_df)), np.full(len(aug_df), aug_weight)])

print("=" * 75)
print("✓ Pemisahan Dataset & Pengayaan Data-Centric Selesai (Zero Data Leakage):")
print(f"  - Train PhiUSIIL Asli      : {len(train_df):,} baris")
print(f"  - Enriched Train Corpus    : {len(aug_df):,} baris (PhishTank+OpenPhish={len(feed_train_phish):,}, Adv={n_adv_phish:,}, Legit/Tranco={n_legit_aug:,})")
print(f"    -> Proporsi Legit HTTP   : {(aug_df[aug_df['target']==0]['URL'].str.startswith('http://').mean()*100):.1f}%")
print(f"    -> Proporsi Phish HTTPS  : {(aug_df[aug_df['target']==1]['URL'].str.startswith('https://').mean()*100):.1f}%")
print(f"  - Total Train v2 Enriched  : {len(train_aug_df):,} baris")
print(f"  - Validation PhiUSIIL Murni: {len(val_df):,} baris")
print(f"  - Test PhiUSIIL Murni      : {len(test_df):,} baris")
print(f"  - Out-of-Time Holdout Baru : {len(oot_df):,} baris ({(oot_df['target']==1).sum():,} Phish terbaru + {(oot_df['target']==0).sum():,} Legit tak terlihat)")
print("=" * 75)

In [ ]:
# [5] Ekstraksi 26 Fitur Tabular/Fuzzy Paralel + Out-of-Fold Char N-Gram TF-IDF (3-5 gram)

workers = os.cpu_count() or 2

def batch_extract(url_series):
    urls_list = url_series.astype(str).tolist()
    with ThreadPoolExecutor(max_workers=workers) as ex:
        rows = list(tqdm(ex.map(extract_v2_features, urls_list), total=len(urls_list)))
    return pd.DataFrame(rows)[V2_TABULAR_COLS]

print("Mengekstrak 26 fitur tabular + fuzzy untuk Train v2, Validation, PhiUSIIL Test, dan Out-of-Time Holdout...")
X_train_v2 = batch_extract(train_aug_df["URL"])
X_val_v2 = batch_extract(val_df["URL"])
X_test_v2 = batch_extract(test_df["URL"])
X_oot_v2 = batch_extract(oot_df["URL"])

y_train_v2 = train_aug_df["target"].values
y_val = val_df["target"].values
y_test = test_df["target"].values
y_oot = oot_df["target"].values

# Fitur v1 (20 fitur murni pada train_df asli tanpa augmentasi) untuk komparasi adil
X_train_v1 = X_train_v2.iloc[:len(train_df)][V1_FEATURE_COLS].copy()
y_train_v1 = train_df["target"].values
X_test_v1 = X_test_v2[V1_FEATURE_COLS].copy()
X_oot_v1 = X_oot_v2[V1_FEATURE_COLS].copy()

# Out-of-Fold Char N-Gram TF-IDF pada URL tanpa prefix http:// / https://
print("Melatih Char N-Gram TF-IDF (3-5 gram, max_features=8000) Out-of-Fold...")
train_stripped = train_aug_df["URL"].map(strip_scheme)
val_stripped = val_df["URL"].map(strip_scheme)
test_stripped = test_df["URL"].map(strip_scheme)
oot_stripped = oot_df["URL"].map(strip_scheme)

oof_tfidf_prob = np.zeros(len(train_aug_df))
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
for tr_idx, va_idx in skf.split(train_stripped, y_train_v2):
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), max_features=8000, min_df=2, sublinear_tf=True)
    clf_tf = LogisticRegression(max_iter=350, C=2.0)
    clf_tf.fit(vec.fit_transform(train_stripped.iloc[tr_idx]), y_train_v2[tr_idx], sample_weight=sample_weights_v2[tr_idx])
    oof_tfidf_prob[va_idx] = clf_tf.predict_proba(vec.transform(train_stripped.iloc[va_idx]))[:, 1]

full_tfidf_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), max_features=8000, min_df=2, sublinear_tf=True)
full_tfidf_clf = LogisticRegression(max_iter=350, C=2.0)
full_tfidf_clf.fit(full_tfidf_vec.fit_transform(train_stripped), y_train_v2, sample_weight=sample_weights_v2)

X_train_v2["tfidf_char_prob"] = oof_tfidf_prob
X_val_v2["tfidf_char_prob"] = full_tfidf_clf.predict_proba(full_tfidf_vec.transform(val_stripped))[:, 1]
X_test_v2["tfidf_char_prob"] = full_tfidf_clf.predict_proba(full_tfidf_vec.transform(test_stripped))[:, 1]
X_oot_v2["tfidf_char_prob"] = full_tfidf_clf.predict_proba(full_tfidf_vec.transform(oot_stripped))[:, 1]

print("✓ Ekstraksi 27 fitur v2 + TF-IDF Out-of-Fold selesai!")

In [ ]:
# [6] Pelatihan Model: Baseline LR vs RF v1 vs PhishGuard v2 (Retrain Enriched)

print("[1/3] Melatih Baseline: Logistic Regression...")
lr_model = LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")
lr_model.fit(X_train_v1, y_train_v1)

print("[2/3] Melatih PhishGuard v1: Random Forest (150 Trees)...")
rf_v1_model = RandomForestClassifier(
    n_estimators=150, criterion="gini", max_features="sqrt",
    min_samples_split=3, class_weight="balanced", random_state=42, n_jobs=-1
)
rf_v1_model.fit(X_train_v1, y_train_v1)

print(f"[3/3] Melatih PhishGuard v2 (Enriched Retrain): {V2_ALGO_NAME}...")
if USE_LIGHTGBM:
    # colsample_bytree membantu mencegah fitur tunggal (seperti has_https) mendominasi seluruh tree
    v2_model = LGBMClassifier(
        n_estimators=400,
        learning_rate=0.06,
        num_leaves=63,
        colsample_bytree=0.85,
        subsample=0.85,
        subsample_freq=1,
        min_child_samples=25,
        reg_lambda=1.5,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
    v2_model.fit(
        X_train_v2, y_train_v2,
        sample_weight=sample_weights_v2,
        eval_set=[(X_val_v2, y_val)],
    )
else:
    v2_model = HistGradientBoostingClassifier(
        max_iter=300,
        learning_rate=0.07,
        max_leaf_nodes=63,
        min_samples_leaf=20,
        l2_regularization=1.5,
        random_state=42
    )
    v2_model.fit(X_train_v2, y_train_v2, sample_weight=sample_weights_v2)

print("✓ Ketiga model selesai dilatih!")

In [ ]:
# [7] Evaluasi Ganda: PhiUSIIL Clean Test + Out-of-Time Real Holdout + 5 Grafik (300 DPI)

def compute_metrics(model, X_eval, y_true):
    prob = model.predict_proba(X_eval)[:, 1]
    pred = (prob >= 0.50).astype(int)
    cm_mat = confusion_matrix(y_true, pred)
    tn_c, fp_c, fn_c, tp_c = cm_mat.ravel()
    return {
        "prob": prob,
        "pred": pred,
        "cm": cm_mat,
        "tn": int(tn_c), "fp": int(fp_c), "fn": int(fn_c), "tp": int(tp_c),
        "fpr": fp_c / (fp_c + tn_c) if (fp_c + tn_c) > 0 else 0.0,
        "fnr": fn_c / (fn_c + tp_c) if (fn_c + tp_c) > 0 else 0.0,
        "acc": accuracy_score(y_true, pred),
        "prec": precision_score(y_true, pred, zero_division=0),
        "rec": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
        "auc": roc_auc_score(y_true, prob),
    }

# Evaluasi pada PhiUSIIL Murni
res_lr = compute_metrics(lr_model, X_test_v1, y_test)
res_v1 = compute_metrics(rf_v1_model, X_test_v1, y_test)
res_v2 = compute_metrics(v2_model, X_test_v2, y_test)

# Evaluasi pada Out-of-Time Holdout Nyata
oot_lr = compute_metrics(lr_model, X_oot_v1, y_oot)
oot_v1 = compute_metrics(rf_v1_model, X_oot_v1, y_oot)
oot_v2 = compute_metrics(v2_model, X_oot_v2, y_oot)

print("=" * 86)
print("      [A] KOMPARASI PADA CLEAN HELD-OUT TEST SET PHIUSIIL")
print("=" * 86)
print(f"{'Model':<38} | {'Acc':>7} | {'Prec':>7} | {'Rec':>7} | {'F1':>7} | {'AUC':>7}")
print("-" * 86)
for label_m, r in [
    ("Logistic Regression (Baseline)", res_lr),
    ("PhishGuard v1 (RF 20 Fitur)", res_v1),
    (f"PhishGuard v2 Enriched", res_v2),
]:
    print(f"{label_m:<38} | {r['acc']*100:6.2f}% | {r['prec']*100:6.2f}% | {r['rec']*100:6.2f}% | {r['f1']*100:6.2f}% | {r['auc']:7.4f}")

print("\n" + "=" * 86)
print(f"      [B] EVALUASI OUT-OF-TIME HOLDOUT ({len(y_oot):,} URL TAK TERLIHAT DI DUNIA NYATA)")
print("=" * 86)
print(f"{'Model':<38} | {'Acc':>7} | {'Prec':>7} | {'Rec':>7} | {'F1':>7} | {'AUC':>7}")
print("-" * 86)
for label_m, r in [
    ("Logistic Regression (Baseline)", oot_lr),
    ("PhishGuard v1 (RF 20 Fitur)", oot_v1),
    ("PhishGuard v2 Enriched", oot_v2),
]:
    print(f"{label_m:<38} | {r['acc']*100:6.2f}% | {r['prec']*100:6.2f}% | {r['rec']*100:6.2f}% | {r['f1']*100:6.2f}% | {r['auc']:7.4f}")
print("=" * 86)
print(f"Out-of-Time Detail v2 -> TN={oot_v2['tn']} | FP={oot_v2['fp']} (FPR={oot_v2['fpr']*100:.2f}%) | FN={oot_v2['fn']} (FNR={oot_v2['fnr']*100:.2f}%) | TP={oot_v2['tp']}")

# Opsional: Jika user mengunggah prev_model_phishing_v2.pkl ke Colab, bandingkan langsung Before vs After!
PREV_PKL_CANDIDATES = ["prev_model_phishing_v2.pkl", "/content/prev_model_phishing_v2.pkl"]
prev_pkl_found = next((p for p in PREV_PKL_CANDIDATES if os.path.isfile(p)), None)
if prev_pkl_found:
    prev_bundle = joblib.load(prev_pkl_found)
    p_cols = prev_bundle["feature_names"]
    X_oot_prev = X_oot_v2.copy()
    if prev_bundle.get("tfidf_vectorizer") and prev_bundle.get("tfidf_classifier"):
        X_oot_prev["tfidf_char_prob"] = prev_bundle["tfidf_classifier"].predict_proba(
            prev_bundle["tfidf_vectorizer"].transform(oot_stripped)
        )[:, 1]
    oot_prev = compute_metrics(prev_bundle["model"], X_oot_prev[p_cols], y_oot)
    print("\n--- Komparasi Langsung Pre-Trained v2 Lama vs Retrain v2 Baru pada Out-of-Time Holdout ---")
    print(f"  v2 Lama : Acc={oot_prev['acc']*100:.2f}% | F1={oot_prev['f1']*100:.2f}% | AUC={oot_prev['auc']:.4f} | FP={oot_prev['fp']} | FN={oot_prev['fn']}")
    print(f"  v2 Baru : Acc={oot_v2['acc']*100:.2f}% | F1={oot_v2['f1']*100:.2f}% | AUC={oot_v2['auc']:.4f} | FP={oot_v2['fp']} | FN={oot_v2['fn']}")

# Grafik 1: Perbandingan Model (PhiUSIIL + Out-of-Time Holdout)
metric_keys = ["acc", "prec", "rec", "f1", "auc"]
metric_labels = ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]
x_pos = np.arange(len(metric_labels))
w = 0.25

plt.figure(figsize=(10, 5))
plt.bar(x_pos - w, [res_lr[k] * 100 for k in metric_keys], w, label="Baseline (LogReg)", color="#94A3B8")
plt.bar(x_pos, [res_v1[k] * 100 for k in metric_keys], w, label="PhishGuard v1 (RF 20 Fitur)", color="#3B82F6")
bars_v2 = plt.bar(x_pos + w, [res_v2[k] * 100 for k in metric_keys], w, label="PhishGuard v2 (Retrain Enriched)", color="#0F766E")
for b in bars_v2:
    h = b.get_height()
    plt.text(b.get_x() + b.get_width() / 2, h + 0.3, f"{h:.2f}%", ha="center", va="bottom", fontsize=8, fontweight="bold", color="#0F766E")
plt.xticks(x_pos, metric_labels, fontweight="bold")
plt.ylabel("Skor (%)")
plt.ylim(75, 104)
plt.title("Perbandingan 3 Generasi Model pada Data Uji PhiUSIIL", fontsize=12, fontweight="bold")
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("v2_model_comparison.png", dpi=300)
plt.show()

# Grafik 2: Confusion Matrix (PhiUSIIL Test)
plt.figure(figsize=(6, 5))
sns.heatmap(
    res_v2["cm"], annot=True, fmt="d", cmap="YlGnBu",
    xticklabels=["Legitimate (0)", "Phishing (1)"], yticklabels=["Legitimate (0)", "Phishing (1)"]
)
plt.title(f"Confusion Matrix PhishGuard v2 ({len(y_test):,} Sampel PhiUSIIL)", fontweight="bold")
plt.xlabel("Prediksi Model")
plt.ylabel("Label Sebenarnya")
plt.tight_layout()
plt.savefig("v2_confusion_matrix.png", dpi=300)
plt.show()

# Grafik 3: Kurva ROC-AUC (PhiUSIIL vs Out-of-Time Holdout)
plt.figure(figsize=(7.5, 5.5))
for lbl, y_ref, r, col, ls in [
    ("PhishGuard v2 (PhiUSIIL)", y_test, res_v2, "#0F766E", "-"),
    ("PhishGuard v2 (Out-of-Time)", y_oot, oot_v2, "#059669", "--"),
    ("PhishGuard v1 (Out-of-Time)", y_oot, oot_v1, "#1D4ED8", ":"),
    ("Baseline LogReg (Out-of-Time)", y_oot, oot_lr, "#F59E0B", "-."),
]:
    fpr_c, tpr_c, _ = roc_curve(y_ref, r["prob"])
    plt.plot(fpr_c, tpr_c, color=col, lw=2.2, linestyle=ls, label=f"{lbl} (AUC = {r['auc']:.4f})")
plt.plot([0, 1], [0, 1], color="#9CA3AF", lw=1.2, linestyle=":")
plt.title("Kurva ROC-AUC: PhiUSIIL vs Out-of-Time Real-World Holdout", fontsize=12, fontweight="bold")
plt.xlabel("False Positive Rate (FPR)")
plt.ylabel("True Positive Rate (TPR)")
plt.grid(linestyle="--", alpha=0.4)
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("v2_roc_curve.png", dpi=300)
plt.show()

# Grafik 4: Permutation Feature Importance (PhiUSIIL + Out-of-Time Holdout)
n_imp = min(3000, len(X_test_v2))
perm_phi = permutation_importance(v2_model, X_test_v2.iloc[:n_imp], y_test[:n_imp], n_repeats=3, random_state=42, n_jobs=1)
v2_importances = pd.Series(perm_phi.importances_mean, index=V2_ALL_COLS).sort_values(ascending=True)

n_oot_imp = min(3000, len(X_oot_v2))
perm_oot = permutation_importance(v2_model, X_oot_v2.iloc[:n_oot_imp], y_oot[:n_oot_imp], n_repeats=3, random_state=42, n_jobs=1)
oot_importances = pd.Series(perm_oot.importances_mean, index=V2_ALL_COLS).sort_values(ascending=True)

print(f"\nShift Dominasi has_https (PhiUSIIL): {v2_importances.get('has_https', 0):.4f} | tfidf_char_prob: {v2_importances.get('tfidf_char_prob', 0):.4f}")
print(f"Top 8 Fitur di Out-of-Time Holdout:")
print(oot_importances.sort_values(ascending=False).head(8).round(4).to_string())

fig, axes = plt.subplots(1, 2, figsize=(15, 7.5))
v2_importances.tail(15).plot(kind="barh", ax=axes[0], color="#0F766E")
axes[0].set_title("Top 15 Feature Importance (PhiUSIIL Test)", fontweight="bold")
axes[0].set_xlabel("Mean Accuracy Decrease")
axes[0].grid(axis="x", linestyle="--", alpha=0.4)

oot_importances.tail(15).plot(kind="barh", ax=axes[1], color="#2563EB")
axes[1].set_title("Top 15 Feature Importance (Out-of-Time Real Holdout)", fontweight="bold")
axes[1].set_xlabel("Mean Accuracy Decrease")
axes[1].grid(axis="x", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("v2_feature_importance.png", dpi=300)
plt.show()

In [ ]:
# [8] Analisis Kesalahan (Error Analysis) pada Out-of-Time Holdout & PhiUSIIL

eval_oot_df = X_oot_v2.copy()
eval_oot_df["url"] = oot_df["URL"].values
eval_oot_df["actual"] = y_oot
eval_oot_df["predicted"] = oot_v2["pred"]
eval_oot_df["phishing_prob"] = np.round(oot_v2["prob"], 4)

tn_df = eval_oot_df[(eval_oot_df["actual"] == 0) & (eval_oot_df["predicted"] == 0)]
fp_df = eval_oot_df[(eval_oot_df["actual"] == 0) & (eval_oot_df["predicted"] == 1)].sort_values("phishing_prob", ascending=False)
fn_df = eval_oot_df[(eval_oot_df["actual"] == 1) & (eval_oot_df["predicted"] == 0)].sort_values("phishing_prob", ascending=True)
tp_df = eval_oot_df[(eval_oot_df["actual"] == 1) & (eval_oot_df["predicted"] == 1)]

print("=" * 75)
print("   ERROR ANALYSIS PHISHGUARD v2 PADA OUT-OF-TIME REAL-WORLD HOLDOUT")
print("=" * 75)
print(f"False Positive (FP): {len(fp_df):,} URL aman tertandai Phishing (FPR = {oot_v2['fpr']*100:.2f}%)")
print(f"False Negative (FN): {len(fn_df):,} URL phishing lolos (FNR = {oot_v2['fnr']*100:.2f}%)")

print("\n--- Contoh 8 Sampel FALSE POSITIVE (Out-of-Time) ---")
for i, (_, row) in enumerate(fp_df.head(8).iterrows(), 1):
    print(f"{i:2d}. [Prob={row['phishing_prob']:.4f} | https={int(row['has_https'])} tfidf={row['tfidf_char_prob']:.3f}] {row['url'][:95]}")

print("\n--- Contoh 8 Sampel FALSE NEGATIVE (Out-of-Time) ---")
for i, (_, row) in enumerate(fn_df.head(8).iterrows(), 1):
    print(f"{i:2d}. [Prob={row['phishing_prob']:.4f} | https={int(row['has_https'])} tfidf={row['tfidf_char_prob']:.3f}] {row['url'][:95]}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
groups = ["TN\n(Aman Benar)", "FP\n(Salah Blokir)", "FN\n(Lolos)", "TP\n(Phishing Benar)"]
means_tfidf = [
    tn_df["tfidf_char_prob"].mean() if len(tn_df) else 0,
    fp_df["tfidf_char_prob"].mean() if len(fp_df) else 0,
    fn_df["tfidf_char_prob"].mean() if len(fn_df) else 0,
    tp_df["tfidf_char_prob"].mean() if len(tp_df) else 0,
]
axes[0].bar(groups, means_tfidf, color=["#10B981", "#F59E0B", "#EF4444", "#0F766E"])
axes[0].set_ylabel("Rata-rata Skor TF-IDF Char N-Gram")
axes[0].set_ylim(0, 1.05)
axes[0].set_title("Kontribusi TF-IDF per Kuadran (Out-of-Time Holdout)", fontweight="bold")
axes[0].grid(axis="y", linestyle="--", alpha=0.4)

if len(fp_df) > 0:
    axes[1].hist(fp_df["phishing_prob"], bins=15, alpha=0.7, color="#F59E0B", label=f"False Positive (n={len(fp_df)})")
if len(fn_df) > 0:
    axes[1].hist(fn_df["phishing_prob"], bins=15, alpha=0.7, color="#DC2626", label=f"False Negative (n={len(fn_df)})")
axes[1].axvline(0.50, color="#111827", linestyle="--", lw=1.8, label="Threshold (0.50)")
axes[1].set_xlim(0.0, 1.0)
axes[1].set_xlabel("Probabilitas Prediksi v2")
axes[1].set_ylabel("Jumlah Sampel")
axes[1].set_title("Distribusi Probabilitas FP & FN (Out-of-Time)", fontweight="bold")
axes[1].grid(axis="y", linestyle="--", alpha=0.4)
axes[1].legend()

plt.tight_layout()
plt.savefig("v2_error_analysis.png", dpi=300)
plt.show()

In [ ]:
# [9] Stress Test Dunia Nyata: 10 URL Klasik + 10 Kasus Sulit Ekstra (Murni Output ML, Tanpa Whitelist)

unseen_hard_urls = [
    ("https://www.liputan6.com/tekno/read/5812345/tips-keamanan-digital-anti-penipuan?utm_source=fb&utm_medium=social", 0, "Aman: Berita path panjang + UTM"),
    ("http://portal.akademik-stikom.ac.id/mhs/krs/cetak", 0, "Aman: HTTP kampus + path"),
    ("https://www.bri.co.id/id/individu/produk-simpanan/britama?ref=homepage_banner", 0, "Aman: Bank resmi path panjang"),
    ("https://docs.python.org/3/library/urllib.parse.html#module-urllib.parse", 0, "Aman: Dokumentasi multi-dot + #"),
    ("https://dana-kaget-gratis-2026.xyz/", 1, "Phishing: HTTPS pendek + Indo lure"),
    ("https://cek-penerima-blt-kemensos.top/ambil", 1, "Phishing: HTTPS + Bansos Indo"),
    ("https://mybca-aktivasi-ulang.com/login", 1, "Phishing: HTTPS + Brand BCA palsu"),
    ("https://g00gle-security-alert.com/verify", 1, "Phishing: HTTPS + Typosquatting g00gle"),
    ("https://paypa1-resolution-center.net/signin", 1, "Phishing: HTTPS + Typosquatting paypa1"),
    ("https://xn--pple-43d.com/account", 1, "Phishing: HTTPS + Homograph Apple"),
]

extended_edge_urls = unseen_hard_urls + [
    ("http://www.kemdikbud.go.id/main/blog/2026/11/panduan-beasiswa-unggulan", 0, "Aman: HTTP .go.id path panjang"),
    ("http://www.ui.ac.id/id/akademik/kalender-kuliah?semester=gasal", 0, "Aman: HTTP .ac.id + query"),
    ("https://www.bni.co.id/id-id/individu/simpanan/taplus-muda?utm_source=ig", 0, "Aman: Bank BNI + UTM"),
    ("https://www.tempo.co/read/1912345/cara-melindungi-akun-m-banking-dari-phishing", 0, "Aman: Berita HTTPS + kata 'phishing'"),
    ("https://www.cnbcindonesia.com/tech/20261125-07-500001/waspada-modus-penipuan", 0, "Aman: Berita panjang normal"),
    ("https://verifikasi-brimo-resmi.pages.dev/login.html", 1, "Phishing: Cloud pages.dev + Indo"),
    ("https://bantuan-sosial-2026.workers.dev/daftar", 1, "Phishing: Cloud workers.dev + Bansos"),
    ("http://194.26.135.42:8080/auth/login.php", 1, "Phishing: IP Host + Port + /login"),
    ("https://103.145.22.91/banking/verify", 1, "Phishing: IP Host HTTPS"),
    ("https://mandlri-livin-support.icu/confirm", 1, "Phishing: Leetspeak mandlri + .icu"),
]

def run_stress(items, title):
    print("=" * 102)
    print(f"  {title}")
    print("=" * 102)
    print(f"{'Deskripsi':<40} | {'Label':<5} | {'Model v1 (RF)':<16} | {'Model v2 Enriched':<18}")
    print("-" * 102)
    h1, h2 = 0, 0
    for u_str, true_lbl, desc in items:
        f_dict = extract_v2_features(u_str)
        x_v1 = pd.DataFrame([[f_dict[c] for c in V1_FEATURE_COLS]], columns=V1_FEATURE_COLS)
        p_v1 = float(rf_v1_model.predict_proba(x_v1)[0][1])
        ok_v1 = int(p_v1 >= 0.50) == true_lbl
        h1 += int(ok_v1)

        f_dict["tfidf_char_prob"] = float(full_tfidf_clf.predict_proba(full_tfidf_vec.transform([strip_scheme(u_str)]))[0][1])
        x_v2 = pd.DataFrame([[f_dict[c] for c in V2_ALL_COLS]], columns=V2_ALL_COLS)
        p_v2 = float(v2_model.predict_proba(x_v2)[0][1])
        ok_v2 = int(p_v2 >= 0.50) == true_lbl
        h2 += int(ok_v2)

        lbl_txt = "PHISH" if true_lbl == 1 else "AMAN "
        print(f"{desc:<40} | {lbl_txt} | {p_v1*100:5.1f}% {'OK' if ok_v1 else 'SALAH':<6} | {p_v2*100:5.1f}% {'OK' if ok_v2 else 'SALAH':<6}")
    print("-" * 102)
    print(f"Total Akurasi -> v1: {h1}/{len(items)} ({h1/len(items)*100:.1f}%) | v2 Enriched: {h2}/{len(items)} ({h2/len(items)*100:.1f}%)\n")
    return h1, h2

v1_hits, v2_hits = run_stress(unseen_hard_urls, "STRESS TEST KLASIK (10 URL SULIT — TANPA WHITELIST)")
v1_ext_hits, v2_ext_hits = run_stress(extended_edge_urls, "EXTENDED EDGE-CASE STRESS TEST (20 URL — CLOUD/IP/HTTP/INDO)")

In [ ]:
# [10] Ekspor Artefak PhishGuard v2 & Auto-Download Google Colab

# 1. Simpan model_phishing_v2.pkl (27 fitur + stacked TF-IDF)
v2_bundle = {
    "model": v2_model,
    "tfidf_vectorizer": full_tfidf_vec,
    "tfidf_classifier": full_tfidf_clf,
    "feature_names": V2_ALL_COLS,
    "tabular_feature_names": V2_TABULAR_COLS,
    "threshold": 0.50,
    "algorithm": V2_ALGO_NAME,
    "metadata": {
        "version": "2.1-data-centric-enriched",
        "dataset": "PhiUSIIL + PhishTank + OpenPhish + Tranco + Indo/HTTP/Cloud Augmentation",
        "features_count": len(V2_ALL_COLS),
        "live_feed_samples_added": len(feed_train_phish),
        "out_of_time_holdout_samples": len(oot_df),
        "sklearn_version": sklearn.__version__,
        "training_environment": "Google Colab",
    }
}
joblib.dump(v2_bundle, "model_phishing_v2.pkl")
print(f"✓ Sukses mengekspor model_phishing_v2.pkl ({os.path.getsize('model_phishing_v2.pkl') / 1e6:.2f} MB)")

# 2. Simpan metrics_v2.json (backward-compatible + metrik Out-of-Time Holdout baru)
feature_rankings = [
    {"feature": feat, "importance": round(float(imp), 4), "rank": i + 1}
    for i, (feat, imp) in enumerate(v2_importances.sort_values(ascending=False).items())
]
oot_feature_rankings = [
    {"feature": feat, "importance": round(float(imp), 4), "rank": i + 1}
    for i, (feat, imp) in enumerate(oot_importances.sort_values(ascending=False).items())
]

metrics_v2 = {
    "dataset": {
        "name": "PhiUSIIL + Enriched Feeds (PhishTank, OpenPhish, Tranco) + Augmentation",
        "phiusiil_unique_samples": len(dataset_df),
        "augmented_train_samples": len(aug_df),
        "live_feed_samples_in_train": len(feed_train_phish),
        "live_openphish_added": len(op_train_urls),
        "out_of_time_holdout_samples": len(oot_df),
        "features_count": len(V2_ALL_COLS),
    },
    "model": {
        "algorithm": V2_ALGO_NAME,
        "threshold": 0.50,
        "sklearn_version": sklearn.__version__,
    },
    "test_metrics_phiusiil": {
        "accuracy": round(float(res_v2["acc"]), 4),
        "precision": round(float(res_v2["prec"]), 4),
        "recall": round(float(res_v2["rec"]), 4),
        "f1_score": round(float(res_v2["f1"]), 4),
        "roc_auc": round(float(res_v2["auc"]), 4),
    },
    "out_of_time_holdout_metrics": {
        "accuracy": round(float(oot_v2["acc"]), 4),
        "precision": round(float(oot_v2["prec"]), 4),
        "recall": round(float(oot_v2["rec"]), 4),
        "f1_score": round(float(oot_v2["f1"]), 4),
        "roc_auc": round(float(oot_v2["auc"]), 4),
        "false_positive_rate": round(float(oot_v2["fpr"]), 6),
        "false_negative_rate": round(float(oot_v2["fnr"]), 6),
        "confusion_matrix": {
            "true_negatives": oot_v2["tn"],
            "false_positives": oot_v2["fp"],
            "false_negatives": oot_v2["fn"],
            "true_positives": oot_v2["tp"],
        },
    },
    "v1_test_metrics_phiusiil": {
        "accuracy": round(float(res_v1["acc"]), 4),
        "precision": round(float(res_v1["prec"]), 4),
        "recall": round(float(res_v1["rec"]), 4),
        "f1_score": round(float(res_v1["f1"]), 4),
        "roc_auc": round(float(res_v1["auc"]), 4),
    },
    "baseline_metrics_phiusiil": {
        "accuracy": round(float(res_lr["acc"]), 4),
        "precision": round(float(res_lr["prec"]), 4),
        "recall": round(float(res_lr["rec"]), 4),
        "f1_score": round(float(res_lr["f1"]), 4),
        "roc_auc": round(float(res_lr["auc"]), 4),
    },
    "stress_test_10_hard_urls": {
        "v1_correct": v1_hits,
        "v2_correct": v2_hits,
        "total": len(unseen_hard_urls),
    },
    "stress_test_20_extended": {
        "v1_correct": v1_ext_hits,
        "v2_correct": v2_ext_hits,
        "total": len(extended_edge_urls),
    },
    "confusion_matrix": {
        "true_negatives": res_v2["tn"],
        "false_positives": res_v2["fp"],
        "false_negatives": res_v2["fn"],
        "true_positives": res_v2["tp"],
        "false_positive_rate": round(float(res_v2["fpr"]), 6),
        "false_negative_rate": round(float(res_v2["fnr"]), 6),
    },
    "feature_importance": feature_rankings,
    "out_of_time_feature_importance": oot_feature_rankings,
}

with open("metrics_v2.json", "w", encoding="utf-8") as f:
    json.dump(metrics_v2, f, indent=2)
print("✓ Sukses mengekspor metrics_v2.json")

# 3. Kemas seluruh 5 grafik v2 ke ZIP
plot_files = [
    "v2_model_comparison.png",
    "v2_confusion_matrix.png",
    "v2_roc_curve.png",
    "v2_feature_importance.png",
    "v2_error_analysis.png",
]
with zipfile.ZipFile("phishguard_v2_evaluation_plots.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for pf in plot_files:
        if os.path.isfile(pf):
            zf.write(pf)
print("✓ Sukses mengemas 5 grafik evaluasi ke phishguard_v2_evaluation_plots.zip")

# 4. Auto-Download di Google Colab
try:
    from google.colab import files
    print("Mengunduh artefak PhishGuard v2 hasil retrain dari Google Colab...")
    files.download("model_phishing_v2.pkl")
    files.download("metrics_v2.json")
    files.download("phishguard_v2_evaluation_plots.zip")
except ImportError:
    print("ℹ Di luar Google Colab: semua file tersedia di direktori kerja saat ini.")